# Routify data generation template

## Environment Setup
Make sure to have a `venv` with the needed dependencies, listet in `jupyter/requirements.txt` set up correctly.

In [ ]:
def reduce(routes):
    results = {}
    for route in routes:
        # Calculate the total distance of the route
        total_distance = sum(e["distance"] for e in route["edges"])
        if total_distance == 0:
            continue  # Skip or handle routes with no distance to avoid division by zero
        
        # Calculate weighted averages for each attribute
        results[route["transportMode"]] = {
            "traveltime": route["traveltime"],
            "distance": total_distance,
            "noise": sum(e["noise"] * e["distance"] for e in route["edges"]) / total_distance,
            "slope": sum(e["slope"] * e["distance"] for e in route["edges"]) / total_distance,
            "greenIndex": sum(e["greenIndex"] * e["distance"] for e in route["edges"]) / total_distance,
            "pm_10": sum(e["pm_10"] * e["distance"] for e in route["edges"]) / total_distance
        }
    return results

## Data generation

Make sure to set the correct path to the GeoJSON file.

In [ ]:
import requests  # pyright: ignore[reportMissingModuleSource]
import geopandas as gpd  # pyright: ignore[reportMissingModuleSource]
import folium  # pyright: ignore[reportMissingImports]
import json
import random
import time
from shapely.geometry import Point  # pyright: ignore[reportMissingModuleSource]
from IPython.display import display

# URL of the GeoJSON file (replace with your desired URL)
with open("PATH_TO_GEOJSON_FILE/boundary_admin_level_8.geojson", "r") as f:
    geojson_data = json.load(f)
    
    # Load the GeoJSON into a GeoDataFrame
    gdf = gpd.GeoDataFrame.from_features(geojson_data["features"], crs="EPSG:4326")
    
    # Reproject to a projected CRS for accurate centroid calculations
    gdf = gdf.to_crs("EPSG:3857")
    centroid = gdf.geometry.centroid.iloc[0]
    
    # Convert back to WGS84 for mapping
    gdf = gdf.to_crs("EPSG:4326")
    centroid = gdf.to_crs("EPSG:4326").geometry.centroid.iloc[0]
    
    # Display the GeoDataFrame
    #display(gdf)
    
    # Create a Folium map centered on the polygon's centroid
    m = folium.Map(location=[centroid.y, centroid.x], zoom_start=6)
    
    # Add the GeoJSON layer
    folium.GeoJson(geojson_data, name="Polygon").add_to(m)
    
    # Display the map
    #display(m)
    
    # Extract polygon boundaries
    polygon = gdf.geometry.unary_union
    minx, miny, maxx, maxy = polygon.bounds
    
    # Function to generate a random point within the polygon
    def generate_random_point_within_polygon():
        while True:
            point = Point(random.uniform(minx, maxx), random.uniform(miny, maxy))
            if polygon.contains(point):
                return point
    
    # Generate 20 pairs of origin and destination coordinates within the polygon
    origins_destinations = [
        {
            "fromLat": (from_point := generate_random_point_within_polygon()).y,
            "fromLon": from_point.x,
            "toLat": (to_point := generate_random_point_within_polygon()).y,
            "toLon": to_point.x,
        }
        for _ in range(20)
    ]

    # API endpoints with different routing modes
    routing_modes = ["routing_mode_distance", "routing_mode_slope", "routing_mode_green", "routing_mode_noise", "routing_mode_air"]
    base_url = "http://localhost:8080/route/"

    successful_responses = []
    total_requests = 10

    while len(successful_responses) < total_requests:
        # Generate a new origin-destination pair
        from_point = generate_random_point_within_polygon()
        to_point = generate_random_point_within_polygon()
        data = {
            "fromLat": from_point.y,
            "fromLon": from_point.x,
            "toLat": to_point.y,
            "toLon": to_point.x,
            "green_index": 50,
            "slope": 50,
            "noise": 50,
            "air": 50
        }
        
        all_responses = {}
        success = True

        for mode in routing_modes:
            route_url = f"{base_url}{mode}/"
            response = requests.post(route_url, json=data)
            
            if response.status_code == 200:
                all_responses[mode] = reduce(response.json())
            else:
                # If any routing mode fails, discard this pair
                success = False
                break

        if success:
            # Store the successful response
            successful_responses.append({"data": data, "responses": all_responses})
            print(str(len(successful_responses)) + "/" + str(total_requests) + " routes requested.")

        # Sleep briefly to avoid overwhelming the server
        time.sleep(0.1)

    # Save the results to a JSON file
    with open("routing_results.json", "w") as f:
        json.dump(successful_responses, f, indent=4)

    print("Successfully saved all route responses.")